# 04 - ECAPA-TDNN Speaker Verification

## Objective

Evaluate whether ECAPA-TDNN embeddings distinguish utterances from the same speaker from utterances spoken by different people.

The experiment builds positive and negative verification trials, extracts or loads ECAPA embeddings, computes cosine-similarity scores, and reports the Equal Error Rate (EER), its threshold, and the corresponding FAR/FRR.

**Protocol**: 100 selected speakers, at least 5 distinct videos per eligible speaker, 10 positive trials and 10 negative trials per selected speaker (2,000 trials in total), with random seed 42.

The trial set is generated once and saved as a reusable experiment artifact. Comparisons between original and anonymized speech must use the exact same speaker pairs and labels. The trial CSV, its SHA-256 hash, the model configuration, and the Git commit should be recorded for each experiment.

Changing the trial-generation implementation, metadata, or configuration may change the sampled pairs. Existing trial sets should therefore be reused rather than regenerated when comparing systems.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from tqdm.auto import tqdm

# Locate the repository root from the notebook's current directory.
cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (
        parent
        for parent in (cwd, *cwd.parents)
        if (parent / "src").is_dir() and (parent / "notebooks").is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Could not locate the repository root.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.paths import (
    VOXCELEB_WAV_DIR,
    METADATA_DIR,
    PROCESSED_DATA_DIR,
)

TRIALS_DIR = PROJECT_ROOT / "data" / "trials"

print("Project root:", PROJECT_ROOT)
print("VoxCeleb WAV directory exists:", VOXCELEB_WAV_DIR.is_dir())
print("Metadata directory exists:", METADATA_DIR.is_dir())

Project root: /Users/juni/Documents/interview-anonymization
VoxCeleb WAV directory exists: True
Metadata directory exists: True


/opt/anaconda3/envs/interview-anon/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from src.verification_trials import resolve_audio_path

metadata_path = METADATA_DIR / "voxceleb_utterances.parquet"
utterance_df = pd.read_parquet(metadata_path)

required_columns = {"speaker_id", "video_id", "path"}
missing_columns = required_columns - set(utterance_df.columns)
if missing_columns:
    raise ValueError(f"Missing metadata columns: {sorted(missing_columns)}")

utterance_df = utterance_df.copy()
utterance_df["path"] = [
    str(resolve_audio_path(path, VOXCELEB_WAV_DIR))
    for path in tqdm(utterance_df["path"], desc="Resolving audio paths")
]

missing_paths = ~utterance_df["path"].map(lambda p: Path(p).is_file())
if missing_paths.any():
    raise FileNotFoundError(
        f"{missing_paths.sum()} audio files are missing. "
        "Check VOXCELEB_WAV_DIR and the metadata paths."
    )

print("Utterances:", len(utterance_df))
print("Speakers:", utterance_df["speaker_id"].nunique())
print("All audio paths exist.")
utterance_df.head()

Resolving audio paths: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 124843/124843 [00:13<00:00, 9297.14it/s]


Utterances: 124843
Speakers: 1166
All audio paths exist.


,speaker_id,video_id,utterance_id,path,split,language,language_confidence
0,id10001,1zcIwhmdeo4,00001,/Users/juni/Documents/interview-anonymization/...,train,english,0.99
1,id10001,1zcIwhmdeo4,00002,/Users/juni/Documents/interview-anonymization/...,train,english,0.99
2,id10001,1zcIwhmdeo4,00003,/Users/juni/Documents/interview-anonymization/...,train,english,0.99
3,id10001,7gWzIy6yIIk,00001,/Users/juni/Documents/interview-anonymization/...,train,english,0.98
4,id10001,7gWzIy6yIIk,00002,/Users/juni/Documents/interview-anonymization/...,train,english,0.98


In [3]:
SEED = 42
MIN_VIDEOS_PER_SPEAKER = 5
N_SPEAKERS = 100
POS_PER_SPEAKER = 10
NEG_PER_SPEAKER = 10


In [4]:
TRIALS_DIR = PROJECT_ROOT / "data" / "trials"
trials_path = TRIALS_DIR / "voxceleb_verification_trials.csv"

verification_trials = pd.read_csv(trials_path)

for column in ["path1", "path2"]:
    verification_trials[column] = verification_trials[column].map(
        lambda path: str(
            resolve_audio_path(path, VOXCELEB_WAV_DIR)
        )
    )

assert len(verification_trials) == 2000
assert verification_trials["label"].isin([0, 1]).all()
assert verification_trials[["path1", "path2"]].notna().all().all()

print("Reference trials loaded:", trials_path)
print("Number of trials:", len(verification_trials))
verification_trials.head()



Reference trials loaded: /Users/juni/Documents/interview-anonymization/data/trials/voxceleb_verification_trials.csv
Number of trials: 2000


,label,path1,path2,speaker1,speaker2
0,0,/Users/juni/Documents/interview-anonymization/...,/Users/juni/Documents/interview-anonymization/...,id10819,id10400
1,1,/Users/juni/Documents/interview-anonymization/...,/Users/juni/Documents/interview-anonymization/...,id10095,id10095
2,0,/Users/juni/Documents/interview-anonymization/...,/Users/juni/Documents/interview-anonymization/...,id10203,id10877
3,1,/Users/juni/Documents/interview-anonymization/...,/Users/juni/Documents/interview-anonymization/...,id10198,id10198
4,0,/Users/juni/Documents/interview-anonymization/...,/Users/juni/Documents/interview-anonymization/...,id10540,id10058


In [5]:
positive = verification_trials[
    verification_trials["label"] == 1
]
negative = verification_trials[
    verification_trials["label"] == 0
]

assert len(positive) == 1000
assert len(negative) == 1000
assert (positive["speaker1"] == positive["speaker2"]).all()
assert (negative["speaker1"] != negative["speaker2"]).all()
assert verification_trials["speaker1"].nunique() == 100
assert verification_trials["speaker2"].nunique() == 100
assert not (
    verification_trials["path1"] == verification_trials["path2"]
).any()

for column in ["path1", "path2"]:
    missing = ~verification_trials[column].map(
        lambda path: Path(path).is_file()
    )
    assert not missing.any(), f"Missing audio files in {column}"

print("Reference trials validated.")

Reference trials validated.


In [6]:
unique_paths = sorted(
    set(verification_trials["path1"])
    | set(verification_trials["path2"])
)

print("Trials:", len(verification_trials))
print("Unique utterances:", len(unique_paths))

Trials: 2000
Unique utterances: 3100


In [7]:
from src.ecapa import extract_embedding, load_ecapa
from src.embedding_cache import SQLiteEmbeddingCache

MODEL_ID = "speechbrain/spkrec-ecapa-voxceleb"

cache = SQLiteEmbeddingCache(
    PROCESSED_DATA_DIR / "ecapa_embeddings.sqlite",
    model_id=MODEL_ID,
)

embedding_cache = {}
computed_now = 0
loaded_from_cache = 0
classifier = None

for audio_path in tqdm(unique_paths, desc="ECAPA embeddings"):
    embedding = cache.get(audio_path)

    if embedding is None:
        if classifier is None:
            classifier = load_ecapa()

        embedding = extract_embedding(audio_path, classifier)
        cache.set(audio_path, embedding)
        computed_now += 1
    else:
        loaded_from_cache += 1

    embedding_cache[audio_path] = embedding

print("Embeddings in memory:", len(embedding_cache))
print("Loaded from SQLite:", loaded_from_cache)
print("Computed in this run:", computed_now)
print("Cache entries for this model:", len(cache))

ECAPA embeddings: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3100/3100 [00:00<00:00, 5362.37it/s]

Embeddings in memory: 3100
Loaded from SQLite: 3100
Computed in this run: 0
Cache entries for this model: 6401


In [8]:
from src.speaker_verification import cosine_similarity

scores = []
for row in tqdm(
    verification_trials.itertuples(index=False),
    total=len(verification_trials),
    desc="Scoring trials",
):
    scores.append(
        cosine_similarity(
            embedding_cache[row.path1],
            embedding_cache[row.path2],
        )
    )

verification_trials = verification_trials.copy()
verification_trials["score"] = scores

verification_trials[
    ["label", "speaker1", "speaker2", "score"]
].head(10)

Scoring trials: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2000/2000 [00:00<00:00, 54568.93it/s]


,label,speaker1,speaker2,score
0,0,id10819,id10400,0.001716
1,1,id10095,id10095,0.635017
2,0,id10203,id10877,0.112272
3,1,id10198,id10198,0.769848
4,0,id10540,id10058,0.000125
5,0,id10972,id10634,0.001138
6,1,id10375,id10375,0.614294
7,0,id10068,id10500,0.360134
8,1,id10222,id10222,0.582467
9,0,id11159,id10401,-0.046516


In [9]:
verification_trials.groupby("label")["score"].describe()

,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
0,1000.0,0.006863,0.091751,-0.277980,-0.052070,0.005504,0.062523,0.363198
1,1000.0,0.624391,0.116312,0.079603,0.553734,0.635148,0.704242,0.976121


In [10]:
from src.speaker_verification import compute_eer

labels = verification_trials["label"].to_numpy()
scores = verification_trials["score"].to_numpy()
eer_results = compute_eer(labels, scores)

print(f"EER: {eer_results['eer'] * 100:.2f}%")
print(f"EER threshold: {eer_results['threshold']:.4f}")
print(f"FAR at EER: {eer_results['far'] * 100:.2f}%")
print(f"FRR at EER: {eer_results['frr'] * 100:.2f}%")

EER: 0.65%
EER threshold: 0.2784
FAR at EER: 0.50%
FRR at EER: 0.80%


In [11]:
import json
from datetime import UTC, datetime
import hashlib
import subprocess

REPORT_DIR = PROJECT_ROOT / "reports" / "speaker_verification"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Save the scored trials.
scored_trials_to_save = verification_trials.copy()

for column in ["path1", "path2"]:
    scored_trials_to_save[column] = scored_trials_to_save[column].map(
        lambda path: Path(path).resolve()
        .relative_to(VOXCELEB_WAV_DIR.resolve())
        .as_posix()
    )

scored_trials_to_save.to_csv(
    REPORT_DIR / "original_scored_trials.csv",
    index=False,
)

# Hash the exact reference-trials file.
trials_hash = hashlib.sha256(trials_path.read_bytes()).hexdigest()

try:
    commit = subprocess.check_output(
        ["git", "rev-parse", "HEAD"],
        cwd=PROJECT_ROOT,
        text=True,
        stderr=subprocess.DEVNULL,
    ).strip()
except (subprocess.CalledProcessError, FileNotFoundError):
    commit = None

report = {
    "condition": "original",
    "timestamp_utc": datetime.now(UTC).isoformat(),
    "model_id": MODEL_ID,
    "git_commit": commit,
    "trials_sha256": trials_hash,
    "n_trials": len(verification_trials),
    "n_unique_audio_files": len(unique_paths),
    "eer": float(eer_results["eer"]),
    "threshold": float(eer_results["threshold"]),
    "far": float(eer_results["far"]),
    "frr": float(eer_results["frr"]),
}

(REPORT_DIR / "original_metrics.json").write_text(
    json.dumps(report, indent=2, allow_nan=False) + "\n",
    encoding="utf-8",
)

447

## Interpretation

A lower EER indicates better speaker-verification performance on this trial set. These results quantify how discriminative ECAPA embeddings are for this protocol; they do not by themselves establish the privacy of an anonymization method.

Record the trial-generation configuration, model version, and EER when comparing experiments. The SQLite cache avoids recomputing embeddings for unchanged audio files under the same `MODEL_ID`.